<a href="https://colab.research.google.com/github/KarlaMichelleSorianoSanhez/Procesos-estocasticos/blob/main/Metodo_de_uniformizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

$$
\color{#002060}{\textbf{Metodo de Uniformización para Cadenas de Markov en Tiempo Continuo}}
$$

**Nombre:** Karla Michelle Soriano Sánchez


**Objetivo** : Implementar el método de uniformización para aproximar la matriz de probabilidades de transición de una cadena de Markov en tiempo continuo y verificar numéricamente la ecuación de Chapman-Kolmogorov.

 ### **Teorema Matriz $P(t)$**

La matriz de probabilidad de transición

$$
P(t)=\big[p_{ij}(t)\big]
$$

está dada por

$$
P(t)
=
\sum_{k=0}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^{\,k}.
$$

Este resultado permite expresar la matriz de transición de una cadena de Markov en tiempo continuo mediante una combinación ponderada de las potencias de la matriz estocástica $\hat P$

### *Aproximación numérica*

Como la serie anterior es infinita, para efectos computacionales se aproxima utilizando únicamente los primeros $M$ términos.

Una elección adecuada para el truncamiento es

$$
M
\approx
\max\{rt+5\sqrt{rt},20\}.
$$




Con este valor se garantiza que los términos omitidos tengan una contribución despreciable y la aproximación obtenida sea suficientemente precisa e la matriz $P(t)$.


$$
\color{#1F4E79}{\textbf{EJERCICIO 3}}
$$

Sea la matriz

$$
R=
\begin{pmatrix}
0 & 2 & 3 & 0\\
4 & 0 & 2 & 0\\
0 & 2 & 0 & 2\\
1 & 0 & 3 & 0
\end{pmatrix}.
$$

1. Use esta propuesta para calcular $P(0.5)$, $P(1)$ y $P(5)$.

2. ¿Se verifica la ecuación de Chapman-Kolmogorov?

$$
P(1)=P(0.5)P(0.5)
$$

### Importación de librerías

Utilizaremos la biblioteca SymPy para realizar los cálculos matriciales de manera simbólica y numérica.

Esto permitirá implementar directamente las expresiones obtenidas en los teoremas demostrados anteriormente.

In [1]:
import sympy as sp

### Definición de la matriz de tasas

La matriz $R=[r_{ij}]$ describe la dinámica de la cadena de Markov en tiempo continuo.

Cada elemento $r_{ij}$ representa la tasa con la que el proceso transita del estado $i$ al estado $j$.

A partir de esta matriz construiremos la matriz estocástica $\hat P requerida por el método de uniformización.

In [2]:
R = sp.Matrix([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
])

R

Matrix([
[0, 2, 3, 0],
[4, 0, 2, 0],
[0, 2, 0, 2],
[1, 0, 3, 0]])

### Cálculo del parámetro r

El método de uniformización requiere seleccionar un número

$$
r \geq \max_i\{r_i\},
$$

donde

$$
r_i=\sum_{j=1}^{N}r_{ij}.
$$

Por lo tanto, primero calculamos la suma de cada renglón de la matriz $R$ y posteriormente elegimos el mayor valor.

In [3]:
def calcular_r(R):
  #sumamos fila a fila para obtener r max
  r_i = [sum(fila) for fila in R.tolist()]

  # Elegimos el valor máximo
  r = max(r_i)

  return r_i, r

In [4]:
r_i, r = calcular_r(R)

print(f"Sumas Individuales (r_i): {r_i}")
print(f"Maxima valor de la suma (r): {r}")

Sumas Individuales (r_i): [5, 6, 4, 4]
Maxima valor de la suma (r): 6


### Construcción de la matriz estocástica $\hat P$

De acuerdo con el teorema de uniformización,

$$
\hat p_{ij}
=
\begin{cases}
1-\dfrac{r_i}{r}, & i=j,\\
\dfrac{r_{ij}}{r}, & i\neq j.
\end{cases}
$$

La matriz $\hat P$ será la matriz de transición de la cadena embebida.

In [5]:
def construir_P_hat(R, r):
  n = R.shape[0]
  P_hat = sp.zeros(n)
  r_i = [sum(fila) for fila in R.tolist()]

  for i in range(n):
    for j in range(n):
      if i == j:
        P_hat[i, j] = 1 - r_i[i]/r

      else:
        P_hat[i, j] = R[i, j]/r

  return sp.Matrix(P_hat)

In [6]:
P_hat = construir_P_hat(R, r)

print("Matriz P_hat:")
display(P_hat)

Matriz P_hat:


Matrix([
[1/6, 1/3, 1/2,   0],
[2/3,   0, 1/3,   0],
[  0, 1/3, 1/3, 1/3],
[1/6,   0, 1/2, 1/3]])

### Verificación de la propiedad estocástica

Antes de continuar verificamos que cada renglón de $\hat P$ sume uno.

Esto garantiza que $\hat P$ es una matriz de transición *válida*.

In [7]:
for i in range(P_hat.rows):
  suma = sum(P_hat[i, j] for j in range(P_hat.cols))
  print(f"Fila {i+1}: {sp.simplify(suma)}")

Fila 1: 1
Fila 2: 1
Fila 3: 1
Fila 4: 1


### Número de términos de truncamiento

El ejercicio propone aproximar la serie infinita utilizando

$$
M \approx \max\{rt+5\sqrt{rt},20\}.
$$

Este valor permite truncar la serie conservando una buena aproximación de $P(t)$.

In [8]:
def calcular_M(r, t):
  rt = r*t
  M = int(sp.ceiling(max(rt + 5*sp.sqrt(rt), 20)))
  return M

### Implementación del Teorema de Uniformización

La matriz de transición está dada por

$$
P(t)
=
\sum_{k=0}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^k.
$$

Como no es posible calcular infinitos términos, utilizamos el truncamiento definido anteriormente y aproximamos

$$
P(t)
\approx
\sum_{k=0}^{M}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^k.
$$

La siguiente función implementa directamente esta expresión.

In [9]:
def calcular_P(t, P_hat, r):
  M = calcular_M(r, t)
  print(f"\nPara t = {t}")
  print(f"M = {M}")

  n = P_hat.shape[0]
  P = sp.zeros(n)

  for k in range(M + 1):
    coeficiente = sp.exp(-r*t) * (r*t)**k / sp.factorial(k)
    P += coeficiente * (P_hat**k)

  return sp.N(P, 12)

### Cálculo de las matrices de transición

Aplicamos el método de uniformización para los tiempos solicitados en el problema.

In [10]:
P05 = calcular_P(0.5, P_hat, r)

print("\nP(0.5)")
display(P05)

P1 = calcular_P(1, P_hat, r)

print("\nP(1)")
display(P1)

P5 = calcular_P(5, P_hat, r)

print("\nP(5)")
display(P5)


Para t = 0.5
M = 20

P(0.5)


Matrix([
[0.250608679645, 0.216964598159, 0.386656935582, 0.145769786602],
[0.253134844845, 0.238360983781, 0.374409239718, 0.134094931645],
[0.169119496516, 0.193614888245, 0.420301017069, 0.216964598159],
[0.158017482467, 0.157444641559, 0.398331790539, 0.286206085423]])


Para t = 1
M = 20

P(1)


Matrix([
[0.206151120157, 0.203902026694, 0.398709595696, 0.191235802346],
[ 0.20828421449, 0.205340699017, 0.397899174557, 0.188474456828],
[ 0.19675849338, 0.198379335659,  0.40095868916, 0.203902026694],
[0.192046223485, 0.193997147863, 0.401470941214, 0.212484232331]])


Para t = 5
M = 58

P(5)


Matrix([
[ 0.19999962635, 0.199999625709,  0.39999924811, 0.199999621199],
[0.199999627061,   0.1999996262,  0.39999924796, 0.199999620146],
[0.199999623304, 0.199999623603, 0.399999248751, 0.199999625709],
[0.199999621348, 0.199999622251, 0.399999249163, 0.199999628605]])

### Verificación de Chapman-Kolmogorov

Para una cadena de Markov en tiempo continuo se cumple

$$
P(t+s)=P(t)P(s).
$$

Tomando

$$
t=s=0.5,
$$

debemos verificar que

$$
P(1)=P(0.5)P(0.5).
$$

In [11]:
producto = P05 * P05

print("P(0.5)P(0.5)")
display(sp.N(producto, 12))

print("P(1)")
display(P1)

P(0.5)P(0.5)


Matrix([
[0.206151411174, 0.203902317711,  0.39871017773, 0.191236093362],
[0.208284505507, 0.205340990034,  0.39789975659, 0.188474747845],
[0.196758784397, 0.198379626676, 0.400959271193, 0.203902317711],
[0.192046514502,  0.19399743888, 0.401471523247, 0.212484523348]])

P(1)


Matrix([
[0.206151120157, 0.203902026694, 0.398709595696, 0.191235802346],
[ 0.20828421449, 0.205340699017, 0.397899174557, 0.188474456828],
[ 0.19675849338, 0.198379335659,  0.40095868916, 0.203902026694],
[0.192046223485, 0.193997147863, 0.401470941214, 0.212484232331]])

### Cálculo del error

Finalmente calculamos la diferencia entre ambas matrices.

Si el error es suficientemente pequeño, podremos concluir que la ecuación de Chapman-Kolmogorov se verifica numéricamente.

In [12]:
error = P1 - producto

print("Error:")
display(sp.N(error, 12))

error_max = max(
    abs(float(error[i, j]))
    for i in range(error.rows)
    for j in range(error.cols)
)

print("Error máximo =", error_max)

Error:


Matrix([
[-2.91016675646e-7, -2.91016704068e-7, -5.82033351293e-7, -2.91016647225e-7],
[-2.91016675646e-7, -2.91016704068e-7, -5.82033351293e-7, -2.91016675646e-7],
[-2.91016647225e-7, -2.91016704068e-7, -5.82033408136e-7, -2.91016647225e-7],
[-2.91016675646e-7, -2.91016675646e-7, -5.82033351293e-7, -2.91016675646e-7]])

Error máximo = 5.820334081363399e-07


### Conclusión

Se implementó exitosamente el método de uniformización para aproximar la matriz de transición de una cadena de Markov en tiempo continuo.

Se calcularon las matrices $P(0.5)$, $P(1)$ y $P(5)$ utilizando el truncamiento sugerido por el ejercicio.

Además, la verificación de Chapman-Kolmogorov mostró un error numérico muy pequeño, confirmando la validez del procedimiento implementado y la consistencia de los resultados obtenidos.

$$
\color{#1F4E79}{\textbf{EJERCICIO 6}}
$$